# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aman-data-search/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)



## 1. My lane as an ML task (type)

My provisional lane is framed as a **Probabilistic Binary Classification and Ranking/Scoring Task**[cite: 3, 4]:

* **Primary Modeling Task (Probabilistic Classification):** The model estimates the posterior probability $P(\text{declining} \mid \mathbf{x})$ that a given content item is experiencing or at high risk of traffic decay[cite: 3, 4].
* **Downstream Operational Task (Ranking & Triage):** The predicted probabilities are combined with visibility and demand signals into a normalized priority score to produce a ranked queue of pages[cite: 3, 4].
* **Why This Framing Fits:** Editorial teams do not have bandwidth to inspect all pages; they operate on a weekly queue capacity (e.g., top 20–50 pages)[cite: 4]. Ranking candidates by predicted decline risk combined with potential exposure directly optimizes editorial time allocation[cite: 3, 4].

In [6]:
import pandas as pd
import numpy as np

# Load starter data
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Verify binary classification label distribution
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
class_counts = df["is_declining_label"].value_counts()
class_pcts = df["is_declining_label"].value_counts(normalize=True) * 100

summary_df = pd.DataFrame({
    "Count": class_counts,
    "Percentage (%)": class_pcts.round(2)
})
summary_df.index = ["Non-Declining (0)", "Declining (1)"]
print("Target Classification Label Distribution:")
print(summary_df)

Target Classification Label Distribution:
                   Count  Percentage (%)
Non-Declining (0)  16262           54.21
Declining (1)      13738           45.79


## 2. Target or proxy

### Target Variable & Proxy Definition
* **Target Label:** Binary indicator `is_declining_label` ($1 = \text{declining}, 0 = \text{non-declining}$).
* **Origin:** In the starter slice, this label is generated directly from `trend_direction == "down"`, which indicates that search impressions in the most recent 30 days fell by more than 20% compared to the prior 30-day window (`trend_pct < -20%`)[cite: 1].
* **Proxy vs. Future Target:** This starter label acts as a historical proxy[cite: 4]. In a production or warehouse implementation, the ideal target is a true forward-looking outcome (e.g., using signals from days 1–90 to predict trajectory in days 91–120)[cite: 4].
* **Leakage Guardrail:** Because `trend_direction` and `trend_pct` directly define the target, they are strictly quarantined and must never enter the model as training features[cite: 1].

In [7]:
# Target verification and leakage source inspection
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Grouped inspection across trend directions
target_summary = df.groupby(["trend_direction", "is_declining_label"]).agg(
    count=("content_id", "count"),
    mean_trend_pct=("trend_pct", "mean"),
    mean_last_30d=("impressions_last_30d", "mean"),
    mean_prev_30d=("impressions_prev_30d", "mean")
).round(2)

print("Target Alignment with Trend Signals (Quarantined Leakage Features):")
print(target_summary)

print("\nTarget Summary:")
print(f"- Total positive (declining) rows: {df['is_declining_label'].sum():,} ({df['is_declining_label'].mean():.2%})")

Target Alignment with Trend Signals (Quarantined Leakage Features):
                                    count  mean_trend_pct  mean_last_30d  \
trend_direction is_declining_label                                         
down            1                   16262          -58.11         941.56   
flat            0                    1152             NaN           0.00   
new             0                    2236             NaN         160.45   
stable          0                    5962           -3.19        2962.58   
up              0                    4388          190.67        2173.77   

                                    mean_prev_30d  
trend_direction is_declining_label                 
down            1                         1808.42  
flat            0                            0.00  
new             0                            0.00  
stable          0                         3105.89  
up              0                         1268.58  

Target Summary:
- Total positive (

## 3. Success metric

### Primary Success Metric: Precision@K (e.g., Precision@50)
* **Why Precision@K Matters Most:** Editorial teams operate under strict weekly review budgets (e.g., auditing 20 to 50 URLs per cycle)[cite: 4]. High ranking precision at the top of the queue ensures editors do not waste hours auditing false positives[cite: 3, 4].
* **Secondary Metrics:** 
  * **Average Precision (PR AUC):** Evaluates overall ranking quality across all thresholds under class imbalance[cite: 3, 4].
  * **ROC AUC:** Measures broad discriminative separation across the full inventory[cite: 3, 4].
* **What "Good" Looks Like:**
  * **Baseline Rule:** Precision@50 $\approx 0.240$, ROC AUC $\approx 0.627$[cite: 4].
  * **Target Threshold:** A viable ML model must achieve **Precision@50 $\ge 0.70$** (a $\sim 3\times$ lift over the rule baseline) and **ROC AUC $\ge 0.75$** on held-out clients[cite: 4].

In [8]:
from sklearn.metrics import precision_score

# Baseline benchmark: Naive sort by raw impressions alone vs True Target
df_sorted = df.sort_values(by="impressions_90d", ascending=False).reset_index(drop=True)
top_50_naive = df_sorted.head(50)
naive_precision_50 = top_50_naive["is_declining_label"].mean()

print(f"Overall Dataset Positive Prevalence (Random Guess Precision): {df['is_declining_label'].mean():.3f}")
print(f"Naive Heuristic Precision@50 (Sort by Impressions only): {naive_precision_50:.3f}")
print(f"Starter Baseline Rule Precision@50 Target: 0.240")
print(f"Minimum ML Acceptance Bar: Precision@50 >= 0.700")

Overall Dataset Positive Prevalence (Random Guess Precision): 0.542
Naive Heuristic Precision@50 (Sort by Impressions only): 0.420
Starter Baseline Rule Precision@50 Target: 0.240
Minimum ML Acceptance Bar: Precision@50 >= 0.700


## 4. The unit of analysis, as a real dataframe

### Unit of Analysis & Data Grain
* **Grain:** One row represents a single pseudonymized content item/page (`content_id`) belonging to a pseudonymized client (`client_id`)[cite: 1].
* **Aggregation Window:** All traffic, search visibility, and user behavior metrics are aggregated over a trailing 90-day window[cite: 1].
* **Feature Representation:** Each observation contains content metadata (word count, age, freshness) alongside 90-day search and engagement metrics (impressions, clicks, CTR, avg position, sessions, engagement rate)[cite: 1].

In [9]:
# Display the unit of analysis: 1 row = 1 pseudonymized content item
sample_cols = [
    "content_id", "client_id", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "sessions_90d", "engagement_rate", "is_declining_label"
]

unit_of_analysis_df = df[sample_cols].head(5)

print(f"Dataset Grain & Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique Content Items (Grain Integrity): {df['content_id'].nunique():,}")
print("\nSample Dataframe (Unit of Analysis):")
display(unit_of_analysis_df)

Dataset Grain & Shape: 30,000 rows × 45 columns
Unique Content Items (Grain Integrity): 30,000

Sample Dataframe (Unit of Analysis):


,content_id,client_id,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,sessions_90d,engagement_rate,is_declining_label
0,content_304f48230142,client_f369cb89fc,187,20,3803,29,0.76,10.6,17,5.88,1
1,content_a1fb4e703a9e,client_4e07408562,445,25,15320,7,0.05,20.3,9,0.00,1
2,content_9aa793d4d895,client_7f2253d7e2,141,20,12581,11,0.09,36.5,11,0.00,1
3,content_331d6c4de07b,client_19581e27de,463,22,11751,58,0.49,6.2,78,1.28,0
4,content_d99b7a2d90ca,client_3fdba35f04,263,14,19140,24,0.13,44.0,145,0.00,1


## 5. Why ML beats a fixed rule here

### Why Rigid Heuristics Fail on Content Inventories
* **Non-Linear Multi-Factor Interactions:** Content decay is rarely driven by a single attribute. A page at position 4 with a 0.4% CTR might be severely underperforming, whereas a page at position 18 with the same CTR is performing above expectations[cite: 4]. Hard-coded threshold rules cannot easily capture position-adjusted baselines or non-linear decay curves[cite: 3, 4].
* **Unranked Binary Output:** A fixed rule (e.g., `impressions >= 500 and days_since_last_update >= 180`) produces a massive unranked boolean bucket containing thousands of URLs without sorting which page offers the highest marginal recovery value[cite: 4].
* **Calibrated Probabilistic Triage:** A trained machine learning model outputs continuous, calibrated probabilities $P(\text{decline} \mid \mathbf{x})$ that weigh search volume, position tiers, engagement rates, and content depth simultaneously[cite: 3, 4]. Combining these probabilities with exposure metrics provides a granular ranking that significantly improves Precision@K over static threshold heuristics[cite: 4].

In [10]:
# Demonstrating heuristic limitations: Single-rule precision vs combined signal messiness
# Rule 1: High Impressions & Stale
r1 = (df["impressions_90d"] >= 500) & (df["days_since_last_update"] >= 180)
# Rule 2: High Impressions & Low Engagement
r2 = (df["impressions_90d"] >= 500) & (df["engagement_rate"] < 30)
# Rule 3: High Impressions & Low CTR in striking distance (pos 1-20)
r3 = (df["impressions_90d"] >= 500) & (df["ctr"] < 0.5) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)

heuristic_results = pd.DataFrame([
    {"Rule": "Stale High-Impression (days >= 180)", "Flagged Count": r1.sum(), "Precision (Actual Declines)": df.loc[r1, "is_declining_label"].mean() if r1.sum() > 0 else 0.0},
    {"Rule": "Low Engagement High-Impression (eng < 30%)", "Flagged Count": r2.sum(), "Precision (Actual Declines)": df.loc[r2, "is_declining_label"].mean() if r2.sum() > 0 else 0.0},
    {"Rule": "Striking Low CTR (pos 1-20, CTR < 0.5%)", "Flagged Count": r3.sum(), "Precision (Actual Declines)": df.loc[r3, "is_declining_label"].mean() if r3.sum() > 0 else 0.0}
])

print("Simple Heuristic Filter Evaluation:")
print(heuristic_results.round(3))

Simple Heuristic Filter Evaluation:
                                         Rule  Flagged Count  \
0         Stale High-Impression (days >= 180)             17   
1  Low Engagement High-Impression (eng < 30%)          16540   
2     Striking Low CTR (pos 1-20, CTR < 0.5%)           9759   

   Precision (Actual Declines)  
0                        0.941  
1                        0.595  
2                        0.627  


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.